In [ ]:
import os
from google.colab import drive
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

drive.mount('/content/drive')

os.environ["GROQ_API_KEY"] = "Add your Groq key here"
db_path = "/content/drive/MyDrive/psych_vector_db"

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if os.path.exists(db_path):
    vector_db = FAISS.load_local(
        db_path,
        embeddings,
        allow_dangerous_deserialization=True
    )
    print("✅ Database successfully loaded from Google Drive.")
else:
    print(f"❌ Error: Could not find '{db_path}'.")

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0
)

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

template = """
You are a professional Psychology Research Assistant at LAU.
Use the Context below to answer the user's Question.

If the answer is not in the Context, tell the user the information isn't in your current database.
Always be concise and helpful.

Context:
{context}

Question: {question}

Helpful Answer:"""

custom_prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vector_db.as_retriever(search_kwargs={"k": 3}),
    memory=memory,
    combine_docs_chain_kwargs={"prompt": custom_prompt}
)

def run_psych_assistant():
    print("\n--- LAU Psychology IPMS with Memory is Active ---")
    while True:
        query = input("\nYou: ")
        if query.lower() in ["exit", "quit", "stop"]:
            print("Shutting down...")
            break

        try:
            response = qa_chain.invoke({"question": query})
            print(f"\nAssistant: {response['answer']}")

        except Exception as e:
            if "rate_limit" in str(e).lower():
                print("⚠️ Rate limit reached on Groq. Please wait 10 seconds.")
            else:
                print(f"⚠️ An error occurred: {e}")

if __name__ == "__main__":
    run_psych_assistant()